# Fine-tune PhoBERT (Showbiz / Giáo dục)

Notebook mỏng: toàn bộ logic nằm trong `ml/phobert/*.py` (config, đọc dữ liệu, chia tập, train). Sửa code ở repo, không sửa ở đây.

Dữ liệu: `labeling data/labeled_*_final.csv` (nhãn đã gộp) (cột `text`, `event_id`, `comment_id`, `verified_label`, ...). Nếu cấu trúc đổi, truyền `--text-col`, `--verified-label-col`... hoặc sửa `phobert_config.py`.

In [ ]:
!pip install -q transformers underthesea openpyxl scikit-learn pandas

In [ ]:
# Lấy code: clone repo hoặc upload thư mục ml/phobert/ lên /content/phobert
# !git clone <repo-url> /content/repo && cp -r /content/repo/ml/phobert /content/phobert
%cd /content/phobert
import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import files
uploaded = files.upload()   # labeled_showbiz_final.csv, labeled_education_final.csv, (tuỳ chọn) tập đánh giá đóng băng

In [ ]:
DOMAIN = "showbiz"   # "showbiz" | "education"
DATA = f"/content/labeled_{DOMAIN}_final.csv"
EVAL = ""            # đường dẫn tập đánh giá đóng băng, để trống nếu chưa có
extra = f"--eval_file {EVAL}" if EVAL else ""
!python phobert_finetune_colab.py --domain {DOMAIN} --data_path {DATA} {extra} --output_dir /content/phobert_checkpoints

In [ ]:
import json
r = json.load(open(f"/content/phobert_checkpoints/{DOMAIN}/test_results.json", encoding="utf-8"))
print("Test F1 macro:", r["test_f1_macro"], "| n_test:", r["n_test"])
import pandas as pd
print(pd.DataFrame(r["confusion_matrix"], index=r["label_order"], columns=r["label_order"]))

In [ ]:
import shutil
shutil.make_archive(f"/content/phobert_{DOMAIN}", "zip", f"/content/phobert_checkpoints/{DOMAIN}")
files.download(f"/content/phobert_{DOMAIN}.zip")

## Gán nhãn hàng loạt (tuỳ chọn)
`!python phobert-batch-inference.py --model_dir /content/phobert_checkpoints/{DOMAIN} --input_path <file> --output_path /content/predictions.csv`